# **`robopianist` tutorial**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/robopianist/blob/main/tutorial.ipynb)








> <p><small><small>Copyright 2023 The RoboPianist Authors.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>

# Installing `robopianist` on Colab

In [1]:
# @title Run to install robopianist
from IPython.display import clear_output
import subprocess

if subprocess.run("nvidia-smi").returncode:
    raise RuntimeError(
        "Cannot communicate with GPU. "
        "Make sure you are using a GPU Colab runtime. "
        "Go to the Runtime menu and select Choose runtime type."
    )

# Install dependencies.
# Note: The original install script uses `sudo apt update`, which will hang in a non-interactive
# Jupyter environment because it waits for a sudo password. We comment it out here.
# If you are missing system dependencies (fluidsynth, portaudio19-dev, ffmpeg), please install them manually in your terminal.
# ! bash <(curl -s https://raw.githubusercontent.com/google-research/robopianist/main/scripts/install_deps.sh) --no-soundfonts --no-menagerie

print("Installing robopianist...")
%pip install -q robopianist>=1.0.6

%env MUJOCO_GL=egl

clear_output()
! echo Installed $(robopianist --version)

Installed robopianist 1.0.10


# Imports

In [2]:
# @title All imports required for this tutorial
from IPython.display import HTML
from base64 import b64encode
import numpy as np
from robopianist.suite.tasks import self_actuated_piano
from robopianist.suite.tasks import piano_with_shadow_hands
from dm_env_wrappers import CanonicalSpecWrapper
from robopianist.wrappers import PianoSoundVideoWrapper
from robopianist import music
from mujoco_utils import composer_utils
import dm_env

In [3]:
# @title Helper functions


# Reference: https://stackoverflow.com/a/60986234.
def play_video(filename: str):
    mp4 = open(filename, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    return HTML(
        """
  <video controls>
        <source src="%s" type="video/mp4">
  </video>
  """
        % data_url
    )

# Self-actuated piano task

In [4]:
task = self_actuated_piano.SelfActuatedPiano(
    midi=music.load("TwinkleTwinkleRousseau"),
    change_color_on_activation=True,
    trim_silence=True,
    control_timestep=0.01,
)

env = composer_utils.Environment(
    recompile_physics=False, task=task, strip_singleton_obs_buffer_dim=True
)

env = PianoSoundVideoWrapper(
    env,
    record_every=1,
    camera_id="piano/back",
    record_dir=".",
)

In [6]:
action_spec = env.action_spec()
min_ctrl = action_spec.minimum
print(min_ctrl)
max_ctrl = action_spec.maximum
print(max_ctrl)
print(f"Action dimension: {action_spec.shape}")

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0.06656816 0.08865588 0.06656816 0.06656816 0.08865588 0.06656816
 0.08865588 0.06656816 0.06656816 0.08865588 0.06656816 0.08865588
 0.06656816 0.08865588 0.06656816 0.06656816 0.08865588 0.06656816
 0.08865588 0.06656816 0.06656816 0.08865588 0.06656816 0.08865588
 0.06656816 0.08865588 0.06656816 0.06656816 0.08865588 0.06656816
 0.08865588 0.06656816 0.06656816 0.08865588 0.06656816 0.08865588
 0.06656816 0.08865588 0.06656816 0.06656816 0.08865588 0.06656816
 0.08865588 0.06656816 0.06656816 0.08865588 0.06656816 0.08865588
 0.06656816 0.08865588 0.06656816 0.06656816 0.08865588 0.06656816
 0.08865588 0.06656816 0.06656816 0.08865588 0.06656816 0.08865588
 0.06656816 0.08865588 0.06656816 0.06656816 0.08865588 0.

In [6]:
print("Observables:")
timestep = env.reset()
dim = 0
for k, v in timestep.observation.items():
    print(f"\t{k}: {v.shape} {v.dtype}")
    dim += np.prod(v.shape)
print(f"Observation dimension: {dim}")

Observables:
	goal: (89,) float64
	piano/activation: (88,) float64
	piano/sustain_activation: (1,) float64
Observation dimension: 178


In [7]:
class Oracle:
    def __call__(self, timestep: dm_env.TimeStep) -> np.ndarray:
        if timestep.reward is not None:
            assert timestep.reward == 0
        # Only grab the next timestep's goal state.
        goal = timestep.observation["goal"][: task.piano.n_keys]
        key_idxs = np.flatnonzero(goal)
        # For goal keys that should be pressed, set the action to the maximum
        # actuator value. For goal keys that should be released, set the action to
        # the minimum actuator value.
        action = min_ctrl.copy()
        action[key_idxs] = max_ctrl[key_idxs]
        # Grab the sustain pedal action.
        action[-1] = timestep.observation["goal"][-1]
        return action

In [8]:
policy = Oracle()

timestep = env.reset()
print(timestep)
while not timestep.last():
    action = policy(timestep)
    # env.step(action) advances the environment by one control timestep (e.g. 0.01s).
    # Under the hood, it calls:
    # 1. task.before_step(physics, action, random_state) -> applies control signals
    # 2. physics.step() -> MuJoCo simulates the physics
    # 3. task.after_step(physics, random_state) -> updates internal states (like t_idx)
    # 4. task.get_reward(physics) -> calculates the reward
    # It returns a `dm_env.TimeStep` object containing:
    # - observation: dict of sensor readings (goal, activation, etc.)
    # - reward: float score for this step
    # - discount: float (usually 1.0, or 0.0 if episode failed)
    # - step_type: enum (FIRST, MID, or LAST)
    # print(timestep.last())
    timestep = env.step(action)

TimeStep(step_type=<StepType.FIRST: 0>, reward=None, discount=None, observation={'goal': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 1.]), 'piano/activation': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.]), 'piano/sustain_activation': array([0.])})


In [9]:
play_video(env.latest_filename)

# Piano with Shadow Hands

In [ ]:
task = piano_with_shadow_hands.PianoWithShadowHands(
    change_color_on_activation=True,
    midi=music.load("TwinkleTwinkleRousseau"),
    trim_silence=True,
    control_timestep=0.05,
    gravity_compensation=True,
    primitive_fingertip_collisions=False,
    reduced_action_space=False,
    n_steps_lookahead=10,
    disable_fingering_reward=False,
    disable_forearm_reward=False,
    disable_colorization=False,
    disable_hand_collisions=False,
    attachment_yaw=0.0,
)

env = composer_utils.Environment(
    task=task, strip_singleton_obs_buffer_dim=True, recompile_physics=False
)

env = PianoSoundVideoWrapper(
    env,
    record_every=1,
    camera_id="piano/back",
    record_dir=".",
)

env = CanonicalSpecWrapper(env)

In [17]:
action_spec = env.action_spec()
print(f"Action dimension: {action_spec.shape}")

min_ctrl_piano_with_shadow_hands = action_spec.minimum
print(min_ctrl_piano_with_shadow_hands)
max_ctrl_piano_with_shadow_hands = action_spec.maximum
print(max_ctrl_piano_with_shadow_hands)
print(f"Action dimension: {action_spec.shape}")

Action dimension: (45,)
[-0.523599 -0.698132 -1.0472    0.       -0.20944  -0.698132 -0.261799
 -0.349066 -0.261799  0.       -0.349066 -0.261799  0.       -0.349066
 -0.261799  0.        0.       -0.349066 -0.261799  0.       -0.7605
  0.       -0.523599 -0.698132 -1.0472    0.       -0.20944  -0.698132
 -0.261799 -0.349066 -0.261799  0.       -0.349066 -0.261799  0.
 -0.349066 -0.261799  0.        0.       -0.349066 -0.261799  0.
 -0.4605    0.        0.      ]
[0.174533 0.488692 1.0472   1.22173  0.20944  0.698132 1.5708   0.349066
 1.5708   3.1415   0.349066 1.5708   3.1415   0.349066 1.5708   3.1415
 0.785398 0.349066 1.5708   3.1415   0.4605   0.06     0.174533 0.488692
 1.0472   1.22173  0.20944  0.698132 1.5708   0.349066 1.5708   3.1415
 0.349066 1.5708   3.1415   0.349066 1.5708   3.1415   0.785398 0.349066
 1.5708   3.1415   0.7605   0.06     1.      ]
Action dimension: (45,)


In [12]:
timestep = env.reset()
dim = 0
for k, v in timestep.observation.items():
    print(f"\t{k}: {v.shape} {v.dtype}")
    dim += int(np.prod(v.shape))
print(f"Observation dimension: {dim}")

	goal: (979,) float64
	fingering: (10,) float64
	piano/state: (88,) float64
	piano/sustain_state: (1,) float64
	rh_shadow_hand/joints_pos: (26,) float64
	lh_shadow_hand/joints_pos: (26,) float64
Observation dimension: 1130


In [13]:
# Download pretrained policy action sequence.
!wget https://github.com/google-research/robopianist/raw/main/examples/twinkle_twinkle_actions.npy > /dev/null 2>&1


class Policy:
    def __init__(self) -> None:
        self.reset()

    def reset(self) -> None:
        self._idx = 0
        self._actions = np.load("twinkle_twinkle_actions.npy")

    def __call__(self, timestep: dm_env.TimeStep) -> np.ndarray:
        del timestep  # Unused.
        actions = self._actions[self._idx]
        self._idx += 1
        return actions

In [14]:
policy = Policy()

timestep = env.reset()
while not timestep.last():
    action = policy(timestep)
    timestep = env.step(action)

In [19]:
play_video(env.latest_filename)